# Evaluation Metrics for Generative Modeling on Spatial Transcriptomics
**Data:** paired spatial-transcriptomics slides (Zhuang ABCA mouse-brain atlas).

A runnable companion to the `paired_slides_eval` suite. The notebook follows one linear workflow:

1. **§1 Generate** the cells to evaluate (per model → a `generated.h5ad`).
2. **§2 Build the shared apparatus** — load the **source / target / classifier** slides and fit
   them into *one* whitened-PCA(50) basis + standardised-coord frame (the NicheFlow recipe).
3. **§3 Train the cell-type classifier** on the classifier slide *(only if you want the `ct/*`
   metrics)*.
4. **§4 Evaluate** — run the whole suite and write per-model CSVs (**§4a**), or run a single metric
   group and just print it (**§4b**).

Everything is scored through the **same** apparatus, so metrics are comparable across models: a
model that emits gene-space cells + raw coords (OT-CFM) is projected/standardised into the exact
basis the niche models (NicheFlow CFM/VFM) already live in — handled automatically by
`evaluate_files(..., shared_pca=True, standardize_coords=...)`. All paths below are **relative to
`notebooks/`**.

## 0. Environment setup

Per-project `uv` venv (run from the `eval_metrics/` repo root):

In [5]:
%%bash
UV_LINK_MODE=copy uv sync --extra pipeline --extra nicheflow --extra classifier 

Resolved 268 packages in 2ms
Checked 245 packages in 60ms


In [8]:
%%bash
cd /work/magroup/shared/enfuzers/generative_modeling/eval_metrics
env -u VIRTUAL_ENV uv run python -m ipykernel install --user \
    --name eval_metrics --display-name "Python (eval_metrics)"

Installed kernelspec eval_metrics in /home/qimow/.local/share/jupyter/kernels/eval_metrics


In [1]:
import sys, torch
print(sys.executable)
print(torch.__version__, torch.version.cuda)

/work/magroup/shared/enfuzers/generative_modeling/eval_metrics/.venv/bin/python3
2.12.1+cu126 12.6


In [1]:
import importlib

import numpy as np
import pandas as pd


def _has(mod):
    try:
        importlib.import_module(mod)
        return True
    except Exception:
        return False


print("optional backends present:")
print("  torch   (MMD / EMD / classifier):", _has("torch"))
print("  ot/pot  (EMD / Wasserstein)     :", _has("ot"))
print("  squidpy (Moran's I)             :", _has("squidpy"))

optional backends present:
  torch   (MMD / EMD / classifier): True
  ot/pot  (EMD / Wasserstein)     : True
  squidpy (Moran's I)             : False


## 1. Generate the cells to evaluate

The metrics compare the real **target slide** against a model's **generated cells**. Generation is
model-specific and already done for the four models below — each stores its cells as a flat/niche
`.h5ad`. The cell just confirms the files are present.

| key | emits | needs shared-PCA replay? |
|---|---|---|
| `otcfm` / `otcfm_spatial` | flat **gene-space** cells + **raw** coords | yes (`shared_pca` + `standardize_coords`) |
| `nicheflow_cfm` / `nicheflow_vfm` | already **whitened X_pca(50)** + **standardised** coords | no (pass-through) |

The next cell just checks the files exist; the one after it **regenerates** a model on the
allocated GPU (run it only if you want fresh cells — it overwrites and needs a checkpoint).

In [ ]:
import os

# Each model's generated cells (one flat/niche .h5ad per model).
ARTIFACTS = {
    "otcfm":          "../artifacts/otcfm_1025/generated.h5ad",               # gene-space + raw coords
    "otcfm_spatial":  "../artifacts/otcfm_spatial_1025/generated.h5ad",       # gene-space + raw coords
    "nicheflow_cfm":  "../artifacts/nicheflow_cfm_unaligned/generated.h5ad",  # whitened X_pca + std coords
    "nicheflow_vfm":  "../artifacts/nicheflow_vfm/generated.h5ad",            # whitened X_pca + std coords
}
for name, path in ARTIFACTS.items():
    print(f"{'OK     ' if os.path.exists(path) else 'MISSING'}  {name:14s} {path}")

In [ ]:
%%bash
# OPTIONAL — (re)generate NicheFlow CFM cells on the allocated GPU (overwrites the artifact; needs a
# checkpoint). The kernel runs inside your SLURM job, so this uses the job's GPU automatically.
# `cd ..` -> repo root, so the paths match what you'd type in a terminal there.
cd ..
python -m paired_slides_eval.generate generator=nicheflow \
    source=../nicheflow_mba/data/adata_Zhuang_Zhuang-ABCA-1.024.h5ad \
    target=../nicheflow_mba/data/adata_Zhuang_Zhuang-ABCA-1.025.h5ad \
    checkpoint=ckpts/NicheFlow_CFM_ABCA.ckpt \
    generated_out=artifacts/nicheflow_cfm_unaligned/generated.h5ad

## 2. Build the shared evaluation apparatus (source · target · classifier slide)

One `preprocess_pair` fit defines the whole measuring apparatus and is reused everywhere:

- **Expression:** `normalize_total → log1p → shared PCA (fit on source+target) → whiten`.
- **Coordinates:** per-slide standardisation `(pos − mean) / std`.

It produces two pickles that carry the *persisted recipe* (`lognorm_mean`, `lognorm_target_sum`,
`var_names`, `PCs`, `stats['X_pca']`, `stats['coords']`):

| file | what it is |
|---|---|
| `../data/abca_pair.pkl` | the apparatus **and** the eval target — source `1.024` → target `1.025` |
| `../data/abca_1.026_clf.pkl` | the held-out **classifier** slide `1.026` projected into the *same* basis |

Slide identities for this dataset: **target = `1.025`** (what every model is scored against),
**generation pair = `1.024 → 1.025`**, **classifier slide = `1.026`** (held out — neither source
nor target). Equivalent one-liner CLI:

```bash
python -m paired_slides_eval.adapters.prepare_shared_slides \
  --source ../../nicheflow_mba/data/adata_Zhuang_Zhuang-ABCA-1.024.h5ad \
  --target ../../nicheflow_mba/data/adata_Zhuang_Zhuang-ABCA-1.025.h5ad \
  --classifier_slide ../../nicheflow_mba/data/adata_Zhuang_Zhuang-ABCA-1.026.h5ad \
  --ct_key class --n_pcs 50 \
  --out_pair ../data/abca_pair.pkl --out_classifier ../data/abca_1.026_clf.pkl
```

In [ ]:
import os
import pickle

from paired_slides_eval.adapters.nicheflow.preprocess import (
    preprocess_classifier_slide,
    preprocess_pair,
)

# --- raw input slides (AnnData .h5ad: raw genes in X + obsm['spatial'] + obs['class']) ---
SLIDES          = "../../nicheflow_mba/data"
SOURCE_H5AD     = f"{SLIDES}/adata_Zhuang_Zhuang-ABCA-1.024.h5ad"   # source (A)
TARGET_H5AD     = f"{SLIDES}/adata_Zhuang_Zhuang-ABCA-1.025.h5ad"   # target (B) = eval target
CLASSIFIER_H5AD = f"{SLIDES}/adata_Zhuang_Zhuang-ABCA-1.026.h5ad"   # held-out classifier-training slide

# --- shared artifacts written here (one fit -> one PCA basis + coord frame) ---
PAIR_PKL = "../data/abca_pair.pkl"         # apparatus + eval target (slide B)
CLF_PKL  = "../data/abca_1.026_clf.pkl"    # classifier slide (1.026) in the same basis
CT_KEY, N_PCS = "class", 50

os.makedirs("../data", exist_ok=True)
if os.path.exists(PAIR_PKL) and os.path.exists(CLF_PKL):
    print(f"using existing {PAIR_PKL}\n                {CLF_PKL}")
else:
    ds_pair, pre = preprocess_pair(SOURCE_H5AD, TARGET_H5AD, n_pcs=N_PCS, cell_type_column=CT_KEY)
    clf_ds = preprocess_classifier_slide(CLASSIFIER_H5AD, pre, cell_type_column=CT_KEY)
    pickle.dump(ds_pair, open(PAIR_PKL, "wb"))
    pickle.dump(clf_ds, open(CLF_PKL, "wb"))
    print(f"built {PAIR_PKL}  (target={ds_pair.timepoints_ordered[-1]}, n_pcs={ds_pair.X_pca.shape[1]})")
    print(f"built {CLF_PKL}  ({len(clf_ds.ct)} cells, {len(clf_ds.ct_ordered)} classes)")

## 3. Train the cell-type classifier *(only if you want the `ct/*` metrics)*

The label-free metrics (§4) run without a classifier. The `concordance` / `ct_gap` groups need one
**`SpatialCTClassifierNet`**, trained on the classifier slide from §2 (`../data/abca_1.026_clf.pkl`)
so it lives in the same basis as the apparatus. This pair has **20** cell types, so override
`data.n_classes=20` (the config default is 17). Train once and reuse the single checkpoint for every
model; the old per-model checkpoints are **incompatible** (different recipe / vocabulary).

The cell below trains on the **allocated GPU** — the kernel runs inside your SLURM job, so
`experiment=classifier/abca_spatial` (which sets `trainer=gpu`) lands on the job's GPU automatically.
The trained checkpoint is copied to `../ckpts/clf_spatial.ckpt`, the path §4a reads.

> **Driver note:** this needs `torch+cu126` (the node's driver is CUDA 12.7). With the current
> `torch+cu130` build it errors with *"driver too old"* — repin to cu126 and `uv sync` first.

In [7]:
%%bash
# Runs on the job's GPU (kernel is inside the SLURM allocation). `cd ..` -> repo root so $PWD and the
# relative out-dirs match a terminal there. Best checkpoint (monitors val/f1) -> ckpts/clf_spatial.ckpt.
cd ..
uv run python -m paired_slides_eval.classifier.train experiment=classifier/abca_spatial \
    data.datamodule.data_fp=$PWD/data/clf.pkl data.n_classes=20 \
    logger=tensorboard hydra.run.dir=outputs/clf_train

task_name: train                                                                
train: true                                                                     
test: true                                                                      
ckpt_path: null                                                                 
seed: '2025'                                                                    
data:                                                                           
  datamodule:                                                                   
    _target_: paired_slides_eval.classifier.datamodule.H5ADCTDataModule         
    split_seed: '2025'                                                          
    train_batch_size: 1024                                                      
    eval_batch_size: 1024                                                       
    num_workers: 2                                                              
    train_ratio: 0.8        

/work/magroup/shared/enfuzers/generative_modeling/eval_metrics/.venv/lib/python3.12/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python3 /work/magroup/shared/enfuzers/generative_modeling/e ...
Using bfloat16 Automatic Mixed Precision (AMP)
Trainer already configured with model summary callbacks: [<class 'lightning.pytorch.callbacks.rich_model_summary.RichModelSummary'>]. Skipping setting a default `ModelSummary` callback.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


[03:19:54][__main__][INFO] [rank: 0] Logging hyperparameters!
[03:19:55][__main__][INFO] [rank: 0] Starting training!


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/work/magroup/shared/enfuzers/generative_modeling/eval_metrics/.venv/lib/python3.12/site-packages/lightning/pytorch/utilities/model_summary/model_summary.py:242: Precision bf16-mixed is not supported by the model summary.  Estimated model size in MB will not be accurate. Using 32 bits instead.


┏━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃    ┃ Name                              ┃ Type       ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0  │ net                               │ SpatialCT… │ 59.6 K │ train │     0 │
│ 1  │ net.point_proj                    │ Linear     │  3.4 K │ train │     0 │
│ 2  │ net.encoder                       │ Sequential │ 33.8 K │ train │     0 │
│ 3  │ net.encoder.0                     │ SAB        │ 16.9 K │ train │     0 │
│ 4  │ net.encoder.0.mab                 │ MAB        │ 16.9 K │ train │     0 │
│ 5  │ net.encoder.0.mab.fc_q            │ Linear     │  4.2 K │ train │     0 │
│ 6  │ net.encoder.0.mab.fc_k            │ Linear     │  4.2 K │ train │     0 │
│ 7  │ net.encoder.0.mab.fc_v            │ Linear     │  4.2 K │ train │     0 │
│ 8  │ net.encoder.0.mab.ln0             │ LayerNorm  │    128 │ train │     0 │
│ 9  │ net.encoder.0.mab.ln1

...

/work/magroup/shared/enfuzers/generative_modeling/eval_metrics/.venv/lib/python3.12/site-packages/torchmetrics/utilities/prints.py:43: UserWarning: No positive samples in targets, true positive value should be meaningless. Returning zero tensor in true positive score
  warnings.warn(*args, **kwargs)


/work/magroup/shared/enfuzers/generative_modeling/eval_metrics/.venv/lib/python3.12/site-packages/lightning/pytorch/loops/fit_loop.py:321: The number of training batches (8) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.


Epoch 0: 100%|██████████| 8/8 [00:00<00:00, 16.97it/s, v_num=0, train/loss=1.830]
Validation: |          | 0/? [00:00<?, ?it/s]
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 1: 100%|██████████| 8/8 [00:00<00:00, 23.48it/s, v_num=0, train/loss=1.470]
Validation: |          | 0/? [00:00<?, ?it/s]
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 2: 100%|██████████| 8/8 [00:00<00:00, 21.38it/s, v_num=0, train/loss=1.200]
Validation: |          | 0/? [00:00<?, ?it/s]
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 3: 100%|██████████| 8/8 [00:00<00:00, 22.59it/s, v_num=0, train/loss=1.090]
Validation: |          | 0/? [00:00<?, ?it/s]
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 4: 100%|██████████| 8/8 [00:00<00:00, 22.15it/s, v_num=0, train/loss=1.200]
Validation: |          | 0/? [00:00<?, ?it/s]
Validation: |          | 0/? [00:00<?, ?it/s]
Epoch 5: 100%|██████████| 8/8 [00:00<00:00, 22.20it/s, v_num=0, train/loss=1.100]
Validation: |          | 0/? [00:00<?, ?it/s]
Va

Restoring states from the checkpoint path at /work/magroup/shared/enfuzers/generative_modeling/eval_metrics/outputs/clf_train/checkpoints/epoch_312.ckpt
Traceback (most recent call last):
  File "/work/magroup/shared/enfuzers/generative_modeling/eval_metrics/src/paired_slides_eval/utils/exceptions.py", line 11, in __call__
    return self.f(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^
  File "/work/magroup/shared/enfuzers/generative_modeling/eval_metrics/src/paired_slides_eval/classifier/train.py", line 71, in main
    trainer.test(model=model, datamodule=datamodule, ckpt_path=ckpt_path)
  File "/work/magroup/shared/enfuzers/generative_modeling/eval_metrics/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/trainer.py", line 821, in test
    return call._call_and_handle_interrupt(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/work/magroup/shared/enfuzers/generative_modeling/eval_metrics/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/call.py", line 

CalledProcessError: Command 'b'# Runs on the job\'s GPU (kernel is inside the SLURM allocation). `cd ..` -> repo root so $PWD and the\n# relative out-dirs match a terminal there. Best checkpoint (monitors val/f1) -> ckpts/clf_spatial.ckpt.\ncd ..\nuv run python -m paired_slides_eval.classifier.train experiment=classifier/abca_spatial \\\n    data.datamodule.data_fp=$PWD/data/clf.pkl data.n_classes=20 \\\n    logger=tensorboard hydra.run.dir=outputs/clf_train\n# mkdir -p ckpts\n# cp "$(ls -t outputs/clf_train/checkpoints/epoch_*.ckpt | head -1)" ckpts/clf_spatial.ckpt\n# echo wrote ckpts/clf_spatial.ckpt\n'' returned non-zero exit status 1.

In [9]:
%%bash
cd ../
cp "$(ls -t outputs/clf_train/checkpoints/epoch_*.ckpt | head -1)" ckpts/clf_spatial.ckpt
echo wrote ckpts/clf_spatial.ckpt

wrote ckpts/clf_spatial.ckpt


## 4. Run the evaluation

This is the same `python -m paired_slides_eval.evaluate` command you'd run in the terminal (a thin
wrapper over `evaluate_files`). Point every model at the **same** `data/abca_pair.pkl`; reconciliation
into the shared space is automatic, so **one command works for all four models**:

- **Expression** — gene-space cells (OT-CFM) are projected into the pickle's whitened shared PCA;
  already-reduced cells (NicheFlow) pass through (dimension auto-detect).
- **Coordinates** — `--coords auto` (default) detects raw vs already-standardised coords and maps
  them into the target frame, logging the decision. No per-model flags.

`--classifier` enables the `ct/*` groups; `--ct_real_reference fixed` makes `ct/acc_real` one
model-independent constant across the table. The `%%bash` cells `cd ..` to the repo root so the paths
match a terminal there.

### 4a. Full metric suite → per-model CSV

Runs the whole suite for each model and writes `reports/<model>/metrics.csv` (the `--out` `metric,value`
layout). The `ct/*` groups are added when the §3 classifier exists, and skip cleanly otherwise.

In [ ]:
%%bash
cd ..
CKPT=ckpts/clf_spatial.ckpt
CLF=""; [ -f "$CKPT" ] && CLF="--classifier $CKPT --ct_real_reference fixed"
[ -z "$CLF" ] && echo "no classifier at $CKPT -> ct/* groups will skip"
METRIC_GROUPS="psd spd distribution c2st c2st_graph moran concordance ct_gap"
for m in otcfm_1025 otcfm_spatial_1025 nicheflow_cfm_unaligned nicheflow_vfm; do
    echo "===== $m ====="
    uv run python -m paired_slides_eval.evaluate \
        --target data/abca_pair.pkl \
        --generated artifacts/$m/generated.h5ad \
        --ct_key class --groups $METRIC_GROUPS $CLF \
        --out reports/$m/metrics_updated.csv
done

===== otcfm_1025 =====


<frozen runpy>:128: RuntimeWarning: 'paired_slides_eval.evaluate' found in sys.modules after import of package 'paired_slides_eval', but prior to execution of 'paired_slides_eval.evaluate'; this may result in unpredictable behaviour
/work/magroup/shared/enfuzers/generative_modeling/eval_metrics/.venv/lib/python3.12/site-packages/docrep/decorators.py:43: SyntaxWarning: 'n_jobs' is not a valid key!
  doc = func(self, args[0].__doc__, *args[1:], **kwargs)
/work/magroup/shared/enfuzers/generative_modeling/eval_metrics/.venv/lib/python3.12/site-packages/docrep/decorators.py:43: SyntaxWarning: 'show_progress_bar' is not a valid key!
  doc = func(self, args[0].__doc__, *args[1:], **kwargs)
/work/magroup/shared/enfuzers/generative_modeling/eval_metrics/src/paired_slides_eval/metrics/morans.py:43: FutureWarning: Calling `spatial_neighbors` is deprecated and will be removed in squidpy v1.9.0. Use `spatial_neighbors_knn`, `spatial_neighbors_radius`, `spatial_neighbors_delaunay`, `spatial_neighbor

INFO     Creating graph using `None` transform and `1` libraries.               


/work/magroup/shared/enfuzers/generative_modeling/eval_metrics/src/paired_slides_eval/metrics/morans.py:43: FutureWarning: Calling `spatial_neighbors` is deprecated and will be removed in squidpy v1.9.0. Use `spatial_neighbors_knn`, `spatial_neighbors_radius`, `spatial_neighbors_delaunay`, `spatial_neighbors_grid`, or `spatial_neighbors_from_builder` instead.
  sq.gr.spatial_neighbors(adata, coord_type="generic", n_neighs=n_neighs)


INFO     Creating graph using `None` transform and `1` libraries.               
target: 9962 cells, 50 features
generated: 9962 cells, 50 feats (flat slide)
test/c2st/acc            0.9985
test/c2st/auc            1.0000
test/c2st/graph_acc      0.9988
test/c2st/graph_auc      1.0000
test/c2st/pos_acc        0.6315
test/ct/acc              0.1167
test/ct/acc_gap          0.2974
test/ct/acc_gen          0.0591
test/ct/acc_real         0.3565
test/ct/f1               0.0307
test/ct/prop_jsd         0.4592
test/ct/prop_kl          9.0205
test/ct/prop_tv          0.8411
test/mmd2/pos            0.0661
test/mmd2/x              0.5434
test/moran/corr          -0.0475
test/moran/gen_mean      -0.0000
test/moran/mae           0.1887
test/moran/real_mean     0.1887
test/ot_w1/pos           0.4311
test/ot_w1/x             7.0950
test/ot_w2/pos           0.5063
test/ot_w2/x             7.2874
test/psd/max             1.1484
test/psd/mean            0.0872
test/spd/max             0.0715
test/spd

<frozen runpy>:128: RuntimeWarning: 'paired_slides_eval.evaluate' found in sys.modules after import of package 'paired_slides_eval', but prior to execution of 'paired_slides_eval.evaluate'; this may result in unpredictable behaviour
/work/magroup/shared/enfuzers/generative_modeling/eval_metrics/.venv/lib/python3.12/site-packages/docrep/decorators.py:43: SyntaxWarning: 'n_jobs' is not a valid key!
  doc = func(self, args[0].__doc__, *args[1:], **kwargs)
/work/magroup/shared/enfuzers/generative_modeling/eval_metrics/.venv/lib/python3.12/site-packages/docrep/decorators.py:43: SyntaxWarning: 'show_progress_bar' is not a valid key!
  doc = func(self, args[0].__doc__, *args[1:], **kwargs)
/work/magroup/shared/enfuzers/generative_modeling/eval_metrics/src/paired_slides_eval/metrics/morans.py:43: FutureWarning: Calling `spatial_neighbors` is deprecated and will be removed in squidpy v1.9.0. Use `spatial_neighbors_knn`, `spatial_neighbors_radius`, `spatial_neighbors_delaunay`, `spatial_neighbor

INFO     Creating graph using `None` transform and `1` libraries.               


/work/magroup/shared/enfuzers/generative_modeling/eval_metrics/src/paired_slides_eval/metrics/morans.py:43: FutureWarning: Calling `spatial_neighbors` is deprecated and will be removed in squidpy v1.9.0. Use `spatial_neighbors_knn`, `spatial_neighbors_radius`, `spatial_neighbors_delaunay`, `spatial_neighbors_grid`, or `spatial_neighbors_from_builder` instead.
  sq.gr.spatial_neighbors(adata, coord_type="generic", n_neighs=n_neighs)


INFO     Creating graph using `None` transform and `1` libraries.               
target: 9962 cells, 50 features
generated: 9962 cells, 50 feats (flat slide)
test/c2st/acc            0.9985
test/c2st/auc            0.9999
test/c2st/graph_acc      0.9988
test/c2st/graph_auc      1.0000
test/c2st/pos_acc        0.4978
test/ct/acc              0.2217
test/ct/acc_gap          0.2855
test/ct/acc_gen          0.0710
test/ct/acc_real         0.3565
test/ct/f1               0.1293
test/ct/prop_jsd         0.2031
test/ct/prop_kl          2.6642
test/ct/prop_tv          0.5779
test/mmd2/pos            -0.0005
test/mmd2/x              0.5297
test/moran/corr          0.3696
test/moran/gen_mean      0.1942
test/moran/mae           0.1140
test/moran/real_mean     0.1887
test/ot_w1/pos           0.0808
test/ot_w1/x             7.0776
test/ot_w2/pos           0.0972
test/ot_w2/x             7.2727
test/psd/max             0.4509
test/psd/mean            0.0237
test/spd/max             0.1788
test/spd/

<frozen runpy>:128: RuntimeWarning: 'paired_slides_eval.evaluate' found in sys.modules after import of package 'paired_slides_eval', but prior to execution of 'paired_slides_eval.evaluate'; this may result in unpredictable behaviour


**Optional** — collate the per-model `metrics.csv` files the CLI just wrote into one wide
comparison table (`reports/model_comparison.csv`)

In [ ]:
import glob
import os

import pandas as pd

cols = {}
for csv_path in sorted(glob.glob("../reports/*/metrics.csv")):
    name = os.path.basename(os.path.dirname(csv_path))
    cols[name] = pd.read_csv(csv_path, index_col="metric")["value"]
table = pd.DataFrame(cols).sort_index()
table.to_csv("../reports/model_comparison.csv")
print("wrote ../reports/model_comparison.csv")
table

### 4b. A single model / subset of groups → print

Drop `--out` and the CLI just prints the metrics (plus the coord auto-decision under `notes:`). Swap
the `--generated` model and `--groups` to inspect a specific metric. Available groups: `psd`, `spd`,
`distribution` (MMD/EMD), `c2st`, `c2st_graph`, `moran`, and — with `--classifier` — `concordance`,
`ct_gap`.

In [ ]:
%%bash
cd ..
python -m paired_slides_eval.evaluate \
    --target data/abca_pair.pkl \
    --generated artifacts/otcfm_spatial_1025/generated.h5ad \
    --ct_key class --groups distribution c2st moran